In [9]:
# =========================
# STAGE 3 — MODEL SCALING (FINAL FINAL)
# =========================

import pandas as pd
import numpy as np
import statsmodels.api as sm
import os
import pickle
from sklearn.metrics import mean_absolute_error

# =========================
# PATHS
# =========================

PROJECT_PATH = r"C:\Users\Harsh Prakash\Desktop\Lending Club Credit Risk Project"
ARTIFACTS_PATH = os.path.join(PROJECT_PATH, "artifacts")

# =========================
# LOAD DATA
# =========================

X_train = pd.read_csv(os.path.join(ARTIFACTS_PATH, "X_train_model.csv"))
X_test  = pd.read_csv(os.path.join(ARTIFACTS_PATH, "X_test_model.csv"))
X_oot   = pd.read_csv(os.path.join(ARTIFACTS_PATH, "X_oot_model.csv"))

y_train = pd.read_csv(os.path.join(ARTIFACTS_PATH, "y_train.csv")).squeeze()
y_test  = pd.read_csv(os.path.join(ARTIFACTS_PATH, "y_test.csv")).squeeze()
y_oot   = pd.read_csv(os.path.join(ARTIFACTS_PATH, "y_oot.csv")).squeeze()

# =========================
# 1. PD ENGINE
# =========================

X_train_sm = sm.add_constant(X_train)
X_test_sm  = sm.add_constant(X_test)
X_oot_sm   = sm.add_constant(X_oot)

result = sm.Logit(y_train, X_train_sm).fit(maxiter=1000)

while True:
    pvals = result.pvalues.drop('const')
    worst = pvals.idxmax()

    if pvals[worst] > 0.05:
        X_train_sm = X_train_sm.drop(columns=[worst])
        X_test_sm  = X_test_sm.drop(columns=[worst])
        X_oot_sm   = X_oot_sm.drop(columns=[worst])
        result = sm.Logit(y_train, X_train_sm).fit(maxiter=1000, disp=False)
    else:
        break

print("\nFINAL MODEL SUMMARY:")
print(result.summary())

final_vars = [c for c in X_train_sm.columns if c != 'const']

# =========================
# PREDICTIONS
# =========================

p_good_train = result.predict(X_train_sm)
p_good_test  = result.predict(X_test_sm)
p_good_oot   = result.predict(X_oot_sm)

pd_train = 1 - p_good_train
pd_test  = 1 - p_good_test
pd_oot   = 1 - p_good_oot

# =========================
# 2. SCORE SCALING
# =========================

PDO = 40
TARGET_SCORE = 600
TARGET_ODDS = 50

Factor = PDO / np.log(2)
Offset = TARGET_SCORE - Factor * np.log(TARGET_ODDS)

def score_fn(p_good):
    odds = p_good / (1 - p_good)
    return (Offset + Factor * np.log(odds)).round()

# 🔴 FIX: restore all score series
score_train = score_fn(p_good_train)
score_test  = score_fn(p_good_test)
score_oot   = score_fn(p_good_oot)

# =========================
# 3. SCORECARD
# =========================

woe_map = {
    'grade_woe': 'woe_grade.csv',
    'dti_woe': 'woe_dti.csv',
    'income_woe': 'woe_income.csv',
    'revol_util_woe': 'woe_revol.csv',
    'delinq_woe': 'woe_delinq.csv',
    'inq_woe': 'woe_inq.csv',
    'home_woe': 'woe_home.csv',
    'purpose_woe': 'woe_purpose.csv',
    'state_woe': 'woe_state.csv',
    'term_woe': 'woe_term.csv'
}

scorecard_rows = []

for var in result.params.index:
    if var == 'const' or var not in woe_map:
        continue

    beta = result.params[var]
    woe_df = pd.read_csv(os.path.join(ARTIFACTS_PATH, woe_map[var]))
    bin_col = woe_df.columns[0]

    for _, row in woe_df.iterrows():
        scorecard_rows.append({
            "Variable": var,
            "Bin": row[bin_col],
            "Points": round(Factor * beta * row["WOE"])
        })

scorecard_df = pd.DataFrame(scorecard_rows)

base_score = round(Offset + Factor * result.params['const'])

# 🔴 FIX: restore score range
var_min = scorecard_df.groupby('Variable')['Points'].min().sum()
var_max = scorecard_df.groupby('Variable')['Points'].max().sum()

print(f"\nBase Score: {base_score}")
print(f"Score Range: {base_score + var_min} to {base_score + var_max}")

# =========================
# VIF CHECK
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = X_train_sm[final_vars]

vif_df = pd.DataFrame({
    'feature': final_vars,
    'VIF': [variance_inflation_factor(X_vif.values, i)
            for i in range(X_vif.shape[1])]
})

print("\nFINAL VIF CHECK:")
print(vif_df.sort_values('VIF', ascending=False))
# =========================
# 4. LGD ENGINE
# =========================

# =========================
# LGD CLEANING (FINAL FIX)
# =========================

df_lgd['int_rate'] = df_lgd['int_rate'].astype(str).str.replace('%','')

for col in ['loan_amnt','int_rate','annual_inc','dti']:
    df_lgd[col] = pd.to_numeric(df_lgd[col], errors='coerce')

df_lgd['annual_inc'] = df_lgd['annual_inc'].fillna(df_lgd['annual_inc'].median())

df_lgd = df_lgd[df_lgd['funded_amnt'] > 0]

df_lgd.replace([np.inf, -np.inf], np.nan, inplace=True)

df_lgd = df_lgd.dropna(subset=[
    'LGD',
    'loan_amnt',
    'int_rate',
    'annual_inc',
    'dti'
]).copy()

df_lgd = df_lgd.reset_index(drop=True)

# 🔴 FIX: explicit time sort
df_lgd['issue_d_dt'] = pd.to_datetime(df_lgd['issue_d'], format='%b-%Y', errors='coerce')
df_lgd = df_lgd.sort_values('issue_d_dt').reset_index(drop=True)

df_lgd['recoveries'] = df_lgd['recoveries'].fillna(0)

df_lgd['LGD'] = (
    (df_lgd['funded_amnt'] - df_lgd['recoveries']) /
    df_lgd['funded_amnt']
).clip(0,1)

df_lgd['annual_inc'] = df_lgd['annual_inc'].fillna(df_lgd['annual_inc'].median())

X_lgd = df_lgd[['loan_amnt','int_rate','annual_inc','dti']]
y_lgd = df_lgd['LGD']

split_idx = int(0.8 * len(X_lgd))

X_lgd_tr = X_lgd.iloc[:split_idx]
X_lgd_te = X_lgd.iloc[split_idx:]

y_lgd_tr = y_lgd.iloc[:split_idx]
y_lgd_te = y_lgd.iloc[split_idx:]



lgd_model = sm.OLS(y_lgd_tr, sm.add_constant(X_lgd_tr)).fit()

lgd_pred_te = np.clip(
    lgd_model.predict(sm.add_constant(X_lgd_te)), 0, 1
)

print("\nLGD MAE:", mean_absolute_error(y_lgd_te, lgd_pred_te))

# =========================
# 5. EAD ENGINE
# =========================

loan_file = os.path.join(ARTIFACTS_PATH, "loan_amnt_test.csv")

if not os.path.exists(loan_file):
    raise ValueError(
        "loan_amnt_test.csv missing. Must be saved in Stage 1."
    )

ead_test = pd.read_csv(loan_file).squeeze().values

# =========================
# EXPECTED LOSS
# =========================

mean_lgd = float(lgd_pred_te.mean())
el = pd_test * mean_lgd * ead_test

print("\nEXPECTED LOSS (PORTFOLIO):")
print("Total EL:", round(el.sum(),2))

print("\nNOTE:")
print("PD: 2014 cohort")
print("LGD: charged-off population across vintages")
print("EAD: loan-level")
print("EL is portfolio-level approximation")

# =========================
# SAVE ARTIFACTS
# =========================

pickle.dump(result, open(os.path.join(ARTIFACTS_PATH, "pd_model.pkl"), "wb"))
pickle.dump(lgd_model, open(os.path.join(ARTIFACTS_PATH, "lgd_model.pkl"), "wb"))
pickle.dump(final_vars, open(os.path.join(ARTIFACTS_PATH, "final_vars_pd.pkl"), "wb"))

# 🔴 FIX: restore full save block

pd.Series(pd_train, name='pd').to_csv(os.path.join(ARTIFACTS_PATH,"pd_train.csv"), index=False)
pd.Series(pd_test,  name='pd').to_csv(os.path.join(ARTIFACTS_PATH,"pd_test.csv"),  index=False)
pd.Series(pd_oot,   name='pd').to_csv(os.path.join(ARTIFACTS_PATH,"pd_oot.csv"),   index=False)

pd.Series(score_train, name='score').to_csv(os.path.join(ARTIFACTS_PATH,"score_train.csv"), index=False)
pd.Series(score_test,  name='score').to_csv(os.path.join(ARTIFACTS_PATH,"score_test.csv"),  index=False)
pd.Series(score_oot,   name='score').to_csv(os.path.join(ARTIFACTS_PATH,"score_oot.csv"),   index=False)

scorecard_df.to_csv(os.path.join(ARTIFACTS_PATH, "scorecard.csv"), index=False)

print("\n✅ STAGE 3 FINAL — FULLY PIPELINE-COMPLETE")

Optimization terminated successfully.
         Current function value: 0.406997
         Iterations 6

FINAL MODEL SUMMARY:
                           Logit Regression Results                           
Dep. Variable:                 target   No. Observations:               230695
Model:                          Logit   Df Residuals:                   230687
Method:                           MLE   Df Model:                            7
Date:                Fri, 05 Jun 2026   Pseudo R-squ.:                 0.06172
Time:                        00:31:40   Log-Likelihood:                -93893.
converged:                       True   LL-Null:                   -1.0007e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const              1.6859      0.006    279.376      0.000       1.674       1